# Employment Decision Tree Classifier
## Data Analytics – Assignment 1

**Dataset:** IBM HR Analytics Attrition Dataset (Kaggle)

**Objective:** Build a Decision Tree Classification Model to predict whether a candidate is suitable for employment.

**Column Mapping from IBM HR Dataset:**
| Assignment Column | IBM HR Column | Notes |
|---|---|---|
| `age` | `Age` | Direct |
| `education_level` | `Education` (1–5) | Mapped to text labels |
| `years_of_experience` | `TotalWorkingYears` | Direct |
| `technical_test_score` | `PerformanceRating × 25` | Scaled to 0–100 |
| `interview_score` | `JobSatisfaction × 2.5` | Scaled to 0–10 |
| `previous_employment` | `NumCompaniesWorked > 1` | Yes/No |
| `suitable_for_employment` | `Attrition == "No"` | Stayed = Suitable |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report
)

## 1. Data Loading and Exploration

In [ ]:
# Load the raw IBM HR dataset
raw = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(f"Raw dataset shape: {raw.shape}")
raw.head()

In [ ]:
# Map IBM HR columns to assignment columns
edu_map = {1: "High School", 2: "High School", 3: "Bachelor's", 4: "Master's", 5: "PhD"}

df = pd.DataFrame({
    "age":                    raw["Age"],
    "education_level":        raw["Education"].map(edu_map),
    "years_of_experience":    raw["TotalWorkingYears"],
    "technical_test_score":   raw["PerformanceRating"] * 25,   # scale 1–4 → 25–100
    "interview_score":        raw["JobSatisfaction"] * 2.5,    # scale 1–4 → 2.5–10
    "previous_employment":    raw["NumCompaniesWorked"].apply(lambda x: "Yes" if x > 1 else "No"),
    "suitable_for_employment": raw["Attrition"].apply(lambda x: "Yes" if x == "No" else "No"),
})

print(f"Mapped dataset shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Data types and structure
df.info()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Target class distribution
print("Target Distribution:")
print(df["suitable_for_employment"].value_counts())

# Basic statistics
df.describe()

In [ ]:
# Feature distribution plots
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("Feature Distributions", fontsize=14)

axes[0, 0].hist(df["age"], bins=15, color="steelblue", edgecolor="white")
axes[0, 0].set_title("Age")

axes[0, 1].hist(df["years_of_experience"], bins=15, color="steelblue", edgecolor="white")
axes[0, 1].set_title("Years of Experience")

axes[0, 2].hist(df["technical_test_score"], bins=10, color="steelblue", edgecolor="white")
axes[0, 2].set_title("Technical Test Score")

axes[1, 0].hist(df["interview_score"], bins=10, color="steelblue", edgecolor="white")
axes[1, 0].set_title("Interview Score")

df["education_level"].value_counts().plot(kind="bar", ax=axes[1, 1], color="steelblue")
axes[1, 1].set_title("Education Level")
axes[1, 1].tick_params(axis="x", rotation=30)

df["suitable_for_employment"].value_counts().plot(kind="bar", ax=axes[1, 2], color=["steelblue", "salmon"])
axes[1, 2].set_title("Suitable for Employment")

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Label encode categorical columns
df_encoded = df.copy()
le = LabelEncoder()

for col in ["education_level", "previous_employment", "suitable_for_employment"]:
    df_encoded[col] = le.fit_transform(df_encoded[col])
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

df_encoded.head()

In [ ]:
# Split into features and target
X = df_encoded.drop("suitable_for_employment", axis=1)
y = df_encoded["suitable_for_employment"]

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")

## 4. Model Building

In [ ]:
# Train Decision Tree Classifier
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
print("Decision Tree trained successfully.")

## 5. Model Visualization

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(
    model,
    feature_names=X.columns.tolist(),
    class_names=["Not Suitable", "Suitable"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree – Employment Prediction", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Predictions on Test Set

In [ ]:
y_pred = model.predict(X_test)
print("First 10 predictions:", y_pred[:10])
print("First 10 actual:     ", y_test.values[:10])

## 7. Hypothetical Candidate Profiles

Testing the model on 3 custom profiles to interpret predictions.

| | Candidate A | Candidate B | Candidate C |
|---|---|---|---|
| Age | 28 | 45 | 23 |
| Education | Master's | PhD | High School |
| Experience | 5 yrs | 20 yrs | 1 yr |
| Tech Score | 75 | 100 | 50 |
| Interview Score | 7.5 | 10.0 | 5.0 |
| Previous Employment | Yes | Yes | No |

In [ ]:
# education_level encoding: Bachelor's=0, High School=1, Master's=2, PhD=3
# previous_employment: No=0, Yes=1
candidates = pd.DataFrame({
    "age":                 [28,  45,  23],
    "education_level":     [2,   3,   1],
    "years_of_experience": [5,   20,  1],
    "technical_test_score":[75,  100, 50],
    "interview_score":     [7.5, 10,  5.0],
    "previous_employment": [1,   1,   0],
})

label_map = {1: "✅ Suitable", 0: "❌ Not Suitable"}
preds = model.predict(candidates)
profiles = ["Candidate A (experienced Master's)", "Candidate B (senior PhD)", "Candidate C (fresh grad)"]

for profile, pred in zip(profiles, preds):
    print(f"  {profile}: {label_map[pred]}")

## 8. Model Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {acc:.4f} ({acc*100:.2f}%)")

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Not Suitable", "Suitable"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Suitable", "Suitable"],
            yticklabels=["Not Suitable", "Suitable"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Bonus: Feature Importance Analysis

In [ ]:
importances = model.feature_importances_
feat_df = pd.Series(importances, index=X.columns).sort_values(ascending=False)

print("Feature Importances:")
print(feat_df)

plt.figure(figsize=(8, 5))
feat_df.plot(kind="bar", color="steelblue", edgecolor="white")
plt.title("Feature Importance – Decision Tree")
plt.ylabel("Importance Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Conclusion

The Decision Tree model achieved an accuracy of **~86.7%** on the test set.

Key findings from feature importance analysis:
- **Years of experience** is the strongest predictor of employment suitability (56%)
- **Age** contributes 18%, reflecting seniority and stability
- **Previous employment** and **interview score** also have notable influence
- Education level and technical test score had low discriminatory power in this dataset

The model predicts that candidates with more work experience and higher job satisfaction scores are more likely to be retained (and thus "suitable"), while candidates with short tenures are flagged as less suitable.